# NOVA RETAIL — Auditoría Inicial de Datos

## Evaluación Forense de Calidad de Datos
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Recepción inicial y revisión de integridad de datos  
**Objetivo:** Evaluar la estructura, calidad y señales iniciales de riesgo en los datos crudos recibidos de NOVA RETAIL.

---

### Alcance de este notebook
Este notebook documenta la primera auditoría forense sobre los datos crudos entregados para el caso NOVA RETAIL. El propósito es:

- verificar disponibilidad y estructura de archivos
- evaluar problemas de calidad de datos en los datasets principales
- identificar riesgos inmediatos de integridad
- establecer una línea base para la reconciliación y detección de anomalías posterior

### Datasets crudos esperados
- `stores.csv`
- `products_sap.csv`
- `products_as400.csv`
- `employees.csv`
- `pos_transactions.csv`

### Nota del analista
Este notebook no busca concluir fraude.  
Este es un notebook de **diagnóstico de integridad de datos y operación**, diseñado para separar señal de ruido antes de cualquier escalamiento investigativo.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

print("Notebook listo.")
print("Ruta de datos:", DATA_PATH.resolve())

Notebook listo.
Ruta de datos: /Users/evelyntorres/nova-retail-loss-prevention/data/raw


In [2]:
stores = pd.read_csv(DATA_PATH / "stores.csv")
products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")
employees = pd.read_csv(DATA_PATH / "employees.csv")
pos = pd.read_csv(DATA_PATH / "pos_transactions.csv")

print("Datasets cargados correctamente.")

Datasets cargados correctamente.


In [3]:
datasets = {
    "stores": stores,
    "products_sap": products_sap,
    "products_as400": products_as400,
    "employees": employees,
    "pos_transactions": pos
}

summary = pd.DataFrame({
    "dataset": datasets.keys(),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()]
})

summary

,dataset,rows,columns
0,stores,187,15
1,products_sap,14000,6
2,products_as400,14000,6
3,employees,2801,10
4,pos_transactions,658851,13


In [4]:
datasets = {
    "stores": stores,
    "products_sap": products_sap,
    "products_as400": products_as400,
    "employees": employees,
    "pos_transactions": pos
}

summary = pd.DataFrame({
    "dataset": list(datasets.keys()),
    "filas": [df.shape[0] for df in datasets.values()],
    "columnas": [df.shape[1] for df in datasets.values()],
    "memoria_mb": [df.memory_usage(deep=True).sum() / 1024**2 for df in datasets.values()]
})

summary.style.format({
    "filas": "{:,}",
    "memoria_mb": "{:.1f}"
}).background_gradient(cmap='viridis')

,dataset,filas,columnas,memoria_mb
0,stores,187,15,0.1
1,products_sap,"14,000",6,4.1
2,products_as400,"14,000",6,3.4
3,employees,"2,801",10,1.1
4,pos_transactions,"658,851",13,284.4


In [5]:
# Distribución de tiendas por sistema
fig = px.pie(
    values=stores['system'].value_counts(),
    names=stores['system'].value_counts().index,
    title="Distribución de tiendas por sistema",
    color_discrete_map={'SAP': '#1f77b4', 'AS400': '#ff7f0e'}
)

fig.show()

In [6]:
# Porcentaje de valores nulos por dataset
null_summary = []

for name, df in datasets.items():
    total_cells = df.shape[0] * df.shape[1]
    total_nulls = df.isna().sum().sum()
    pct_nulls = (total_nulls / total_cells) * 100 if total_cells > 0 else 0

    null_summary.append({
        "dataset": name,
        "total_celdas": total_cells,
        "valores_nulos": total_nulls,
        "porcentaje_nulos": pct_nulls
    })

null_summary_df = pd.DataFrame(null_summary)
null_summary_df.style.format({
    "total_celdas": "{:,}",
    "valores_nulos": "{:,}",
    "porcentaje_nulos": "{:.2f}%"
}).background_gradient(subset=["porcentaje_nulos"], cmap="Reds")

,dataset,total_celdas,valores_nulos,porcentaje_nulos
0,stores,"2,805",0,0.00%
1,products_sap,"84,000",0,0.00%
2,products_as400,"84,000","3,422",4.07%
3,employees,"28,010","1,691",6.04%
4,pos_transactions,"8,565,063","201,064",2.35%


In [7]:
ghost_stores = stores[stores["ghost_store"] == True]

print(f"Total de ghost stores: {len(ghost_stores)}")
ghost_stores[[
    "store_id", "store_name", "city", "state", "system",
    "cedis_route", "regional_manager_id"
]].sort_values("store_id")

Total de ghost stores: 3


,store_id,store_name,city,state,system,cedis_route,regional_manager_id
144,145,NOVA-SAN-145,San Nicolás,Nuevo León,AS400,NORTE,GR-001
170,171,NOVA-MAZ-171,Mazatlán,Sinaloa,AS400,NORTE,GR-002
179,180,NOVA-HER-180,Hermosillo,Sonora,AS400,NORTE,GR-001


In [8]:
as400_quality = {
    "total_registros": len(products_as400),
    "sku_as400_nulos": products_as400["sku_as400"].isna().sum(),
    "description_as400_nulos": products_as400["description_as400"].isna().sum(),
    "category_as400_vacias": (products_as400["category_as400"] == "").sum(),
    "price_as400_nulos": products_as400["price_as400"].isna().sum(),
    "apple_ghosts": products_as400["is_ghost"].sum()
}

pd.DataFrame(as400_quality.items(), columns=["metrica", "valor"])

,metrica,valor
0,total_registros,14000
1,sku_as400_nulos,22
2,description_as400_nulos,22
3,category_as400_vacias,0
4,price_as400_nulos,589
5,apple_ghosts,22


In [9]:
pos_nulls = (
    pos.isna()
    .sum()
    .reset_index()
)

pos_nulls.columns = ["columna", "nulos"]
pos_nulls["porcentaje"] = (pos_nulls["nulos"] / len(pos)) * 100
pos_nulls = pos_nulls.sort_values("nulos", ascending=False)

pos_nulls.style.format({
    "nulos": "{:,}",
    "porcentaje": "{:.2f}%"
}).background_gradient(subset=["porcentaje"], cmap="Reds")

,columna,nulos,porcentaje
5,discount_pct,"201,064",30.52%
0,transaction_id,0,0.00%
1,store_id,0,0.00%
2,sku,0,0.00%
3,quantity,0,0.00%
4,unit_price,0,0.00%
6,total,0,0.00%
7,timestamp,0,0.00%
8,payment_method,0,0.00%
9,cashier_id,0,0.00%


In [10]:
as400_nulls = (
    products_as400.isna()
    .sum()
    .reset_index()
)

as400_nulls.columns = ["columna", "nulos"]
as400_nulls["porcentaje"] = (as400_nulls["nulos"] / len(products_as400)) * 100
as400_nulls = as400_nulls.sort_values("nulos", ascending=False)

as400_nulls.style.format({
    "nulos": "{:,}",
    "porcentaje": "{:.2f}%"
}).background_gradient(subset=["porcentaje"], cmap="Oranges")

,columna,nulos,porcentaje
2,category_as400,"2,789",19.92%
3,price_as400,589,4.21%
0,sku_as400,22,0.16%
1,description_as400,22,0.16%
4,sku_sap_reference,0,0.00%
5,is_ghost,0,0.00%


In [11]:
# Conversión de timestamp sin intentar renderizar demasiado
pos["timestamp"] = pd.to_datetime(pos["timestamp"], errors="coerce")

year_counts = pos["timestamp"].dt.year.value_counts(dropna=False).sort_index()
print(year_counts)

timestamp
1970      3315
2024    655536
Name: count, dtype: int64


In [13]:
timestamps_1970_sample = pos.loc[
    pos["timestamp"].dt.year == 1970,
    ["transaction_id", "store_id", "sku", "timestamp", "system_source", "sync_status"]
].head(10)

timestamps_1970_sample

,transaction_id,store_id,sku,timestamp,system_source,sync_status
16,TX-000000017,1,SAP-0005730,1970-01-01 10:29:00,SAP,SYNCED
547,TX-000000548,1,SAP-0011533,1970-01-01 14:10:00,SAP,SYNCED
656,TX-000000657,1,SAP-0004890,1970-01-01 15:25:00,SAP,SYNCED
823,TX-000000824,1,SAP-0001695,1970-01-01 19:40:00,SAP,SYNCED
1151,TX-000001152,1,SAP-0013112,1970-01-01 18:12:00,SAP,SYNCED
1374,TX-000001375,1,SAP-0012817,1970-01-01 15:37:00,SAP,SYNCED
1816,TX-000001817,1,SAP-0001666,1970-01-01 19:05:00,SAP,SYNCED
1958,TX-000001959,1,SAP-0013506,1970-01-01 14:08:00,SAP,OFFLINE_PENDING
2019,TX-000002020,1,SAP-0005174,1970-01-01 21:59:00,SAP,SYNCED
2102,TX-000002103,1,SAP-0010880,1970-01-01 16:47:00,SAP,OFFLINE_PENDING


In [14]:
pos_nulls = (
    pos.isna()
    .sum()
    .reset_index()
)

pos_nulls.columns = ["columna", "nulos"]
pos_nulls["porcentaje"] = (pos_nulls["nulos"] / len(pos)) * 100
pos_nulls = pos_nulls.sort_values("nulos", ascending=False)

pos_nulls

,columna,nulos,porcentaje
5,discount_pct,201064,30.51737
0,transaction_id,0,0.00000
1,store_id,0,0.00000
2,sku,0,0.00000
3,quantity,0,0.00000
4,unit_price,0,0.00000
6,total,0,0.00000
7,timestamp,0,0.00000
8,payment_method,0,0.00000
9,cashier_id,0,0.00000


In [15]:
as400_nulls = (
    products_as400.isna()
    .sum()
    .reset_index()
)

as400_nulls.columns = ["columna", "nulos"]
as400_nulls["porcentaje"] = (as400_nulls["nulos"] / len(products_as400)) * 100
as400_nulls = as400_nulls.sort_values("nulos", ascending=False)

as400_nulls

,columna,nulos,porcentaje
2,category_as400,2789,19.921429
3,price_as400,589,4.207143
0,sku_as400,22,0.157143
1,description_as400,22,0.157143
4,sku_sap_reference,0,0.000000
5,is_ghost,0,0.000000
